In [ ]:
import os 
os.getcwd()
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
import os

In [ ]:

# Load the filez
data = loadmat('data.mat')
illumination = loadmat("illumination.mat")
pose=loadmat("pose.mat")
# See all the keys (variables stored in the .mat file)
print(f"data keys are {data.keys()}")
print(f"illumination keys are {illumination.keys()}")
print(f"pose keys are {pose.keys()}")

In [ ]:
n=np.arange(1,201,1)
data_classified = {}
for i in n:
    imgs=[]
    for j in range(3):
        imgs.append(data["face"][:,:,3*i-3+j])
    data_classified[i]=imgs

In [ ]:
n_2=np.arange(0,68,1)
n_2p=np.arange(0,13,1)
pose_classified={}
for i in n_2:
    imgs=[]
    for j in n_2p:
        imgs.append(pose["pose"][:,:,j,i])
    pose_classified[i]=imgs

In [ ]:
n_3=np.arange(0,68,1)
n_3i=np.arange(0,21,1)
illum_classified={}
for i in n_3:
    imgs=[]
    for j in n_3i:
        imgs.append(illumination["illum"][:,j,i])
    illum_classified[i]=imgs

# data + labeling

In [ ]:
flattened_data = np.array([data["face"][:, :, i].flatten() for i in range(600)])
person_labels = np.repeat(np.arange(200), 3) #for task1

# for task2:
neutral_indices = list(range(0, 600, 3))
expression_indices = list(range(1, 600, 3))
binary_labels = np.zeros(600, dtype=int)
binary_labels[expression_indices] = 1


# PCA_funciton

In [ ]:
import numpy as np

def compute_pca(data_matrix, num_components=None, variance_threshold=None):
    """
    Perform PCA on a data matrix (samples × features).
    
    Parameters:
        data_matrix: np.ndarray, shape (n_samples, n_features)
        num_components: int or None — number of components to keep
        variance_threshold: float or None — keep components that explain up to this cumulative variance (e.g., 0.95)
    
    Returns:
        pca_result: projected data, shape (n_samples, num_components)
        components: principal components (eigenvectors)
        explained_variance_ratio: array of variance explained by each component
        mean: mean of original data (for inverse transform if needed)
    """
    # Step 1: Center the data
    mean = np.mean(data_matrix, axis=0)
    centered_data = data_matrix - mean

    # Step 2: Covariance matrix
    cov_matrix = np.cov(centered_data, rowvar=False)

    # Step 3: Eigen decomposition
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

    # Step 4: Sort in descending order
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    # Step 5: Compute explained variance
    explained_variance_ratio = eigenvalues / np.sum(eigenvalues)
    cumulative_variance = np.cumsum(explained_variance_ratio)

    # Step 6: Determine number of components
    if variance_threshold is not None:
        num_components = np.argmax(cumulative_variance >= variance_threshold) + 1
        print(f"Using {num_components} components to explain {variance_threshold*100:.1f}% variance.")
    elif num_components is None:
        num_components = data_matrix.shape[1]  # Keep all

    # Step 7: Select components and project
    selected_components = eigenvectors[:, :num_components]
    pca_result = np.dot(centered_data, selected_components)

    return pca_result, selected_components, explained_variance_ratio[:num_components], mean


# MDA Function

In [ ]:
import numpy as np

def compute_mda(data_matrix, labels, num_components=None):
    """
    Perform MDA (also known as LDA) on the given data.

    Parameters:
        data_matrix: np.ndarray of shape (n_samples, n_features)
        labels: array-like of shape (n_samples,)
        num_components: int or None — number of components to retain (must be ≤ n_classes - 1)

    Returns:
        mda_result: Projected data of shape (n_samples, num_components)
        components: Eigenvectors used for projection (n_features, num_components)
        eigenvalues: Corresponding eigenvalues
        overall_mean: Mean of the original data
    """
    n_samples, n_features = data_matrix.shape
    unique_classes = np.unique(labels)
    n_classes = len(unique_classes)

    # Step 1: Center the data
    overall_mean = np.mean(data_matrix, axis=0)
    centered_data = data_matrix - overall_mean

    # Step 2: Compute class means
    class_means = []
    for c in unique_classes:
        class_data = data_matrix[labels == c]
        class_mean = np.mean(class_data, axis=0)
        class_means.append(class_mean)

    # Step 3: Compute between-class scatter matrix (S_B)
    S_B = np.zeros((n_features, n_features))
    for i, c in enumerate(unique_classes):
        n_i = np.sum(labels == c)
        mean_diff = (class_means[i] - overall_mean).reshape(-1, 1)
        S_B += (n_i / n_samples) * (mean_diff @ mean_diff.T)

    # Step 4: Compute within-class scatter matrix (S_W)
    S_W = np.zeros((n_features, n_features))
    for i, c in enumerate(unique_classes):
        class_data = data_matrix[labels == c]
        n_i = class_data.shape[0]
        class_centered = class_data - class_means[i]
        S_W += (n_i / n_samples) * (class_centered.T @ class_centered) / n_i

    # Step 5: Solve generalized eigenvalue problem
    S_W_inv = np.linalg.pinv(S_W)
    eig_matrix = S_W_inv @ S_B
    eigenvalues, eigenvectors = np.linalg.eigh(eig_matrix)

    # Step 6: Sort eigenvectors by eigenvalue magnitude (descending)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    # Step 7: Limit number of components
    max_components = n_classes - 1
    if num_components is None or num_components > max_components:
        num_components = max_components
        print(f"Using max possible components for MDA: {num_components}")

    selected_components = eigenvectors[:, :num_components]
    mda_result = centered_data @ selected_components

    return mda_result, selected_components, eigenvalues[:num_components], overall_mean


# data seperation

In [ ]:
def separate_train_test_manual(data, labels, sub_num_total, train_num, task_num, random_state=42):
    np.random.seed(random_state)
    
    if task_num == 1:  # Person identification with all 3 images
        selected_subjects = np.random.choice(sub_num_total, train_num, replace=False)
        train_indices = []
        test_indices = []

        # For each subject, put 2 images in training and 1 in testing
        for s in selected_subjects:
            base = 3 * s
            train_indices.extend([base, base + 1])  # First 2 images to training
            test_indices.append(base + 2)           # Last image to testing

        train_set = data[train_indices]
        train_labels = labels[train_indices]
        test_set = data[test_indices]
        test_labels = labels[test_indices]

    elif task_num == 2:
        # Get indices for neutral and expression images
        neutral_indices = list(range(0, 3 * sub_num_total, 3))     # Images at positions 0, 3, 6, ...
        expression_indices = list(range(1, 3 * sub_num_total, 3))  # Images at positions 1, 4, 7, ...
        
        # Create binary labels (0 for neutral, 1 for expression)
        neutral_labels = np.zeros(len(neutral_indices), dtype=int)
        expression_labels = np.ones(len(expression_indices), dtype=int)
        
        # Shuffle each set of indices separately
        np.random.shuffle(neutral_indices)
        np.random.shuffle(expression_indices)
        
        # Split each class into training (80%) and testing (20%)
        neutral_split = int(len(neutral_indices) * 0.8)
        expression_split = int(len(expression_indices) * 0.8)
        
        # Create training and testing sets for each class
        neutral_train = neutral_indices[:neutral_split]
        neutral_test = neutral_indices[neutral_split:]
        expression_train = expression_indices[:expression_split]
        expression_test = expression_indices[expression_split:]
        
        # Combine indices and labels
        train_indices = np.concatenate([neutral_train, expression_train])
        test_indices = np.concatenate([neutral_test, expression_test])
        
        # Create labels matching the indices
        train_labels = np.concatenate([np.zeros(len(neutral_train)), np.ones(len(expression_train))])
        test_labels = np.concatenate([np.zeros(len(neutral_test)), np.ones(len(expression_test))])
        
        # Extract the data
        train_set = data[train_indices]
        test_set = data[test_indices]
    
    else:
        raise ValueError("❌ Invalid task_num. Use 1 for person ID or 2 for expression classification.")
    return train_set, train_labels, test_set, test_labels


# K-NN

In [ ]:
import numpy as np
from scipy.stats import mode
from collections import Counter

def my_knn(k, train_set, test_set, train_labels, test_labels, task_num=1, plot_flag=False):
    """
    K-Nearest Neighbors classifier
    
    Parameters:
        k: int, number of neighbors
        train_set: training data
        test_set: test data
        train_labels: training labels
        test_labels: test labels
        task_num: 1 for person ID, 2 for expression classification
        plot_flag: whether to plot results
        
    Returns:
        knn_out: predicted labels
        accuracy: classification accuracy in percentage
    """
    n_test = len(test_set)
    knn_out = np.zeros(n_test, dtype=int)
    
    for i_test in range(n_test):
        # Calculate distances to all training samples
        distances = np.linalg.norm(test_set[i_test] - train_set, axis=1)
        
        # Find k nearest neighbors
        k_nearest_indices = np.argsort(distances)[:k]
        k_nearest_labels = train_labels[k_nearest_indices]
        k_nearest_distances = distances[k_nearest_indices]
        
        # Count occurrences of each label
        label_counts = Counter(k_nearest_labels)
        
        # Get the most common labels and their counts
        most_common = label_counts.most_common()
#         print(f"most common : {most_common}")
#         print(f"most common[0] {most_common[0]}")

        # Check if there's a tie for the most common label
        if len(most_common) > 1 and most_common[0][1] == most_common[1][1]:
#             print(f"most common[1] {most_common[1]}")
            # There's a tie - resolve by closest distance
            tied_labels = [label for label, count in most_common if count == most_common[0][1]]
            
#             Find the tied label with the closest neighbor
#             min_distance = float('inf')
#             closest_label = None
            
#             for idx, label in enumerate(k_nearest_labels):
#                 if label in tied_labels and k_nearest_distances[idx] < min_distance:
#                     min_distance = k_nearest_distances[idx]
#                     closest_label = label
            avg_distances = {}
            for label in tied_labels:
                label_distances = [
                    k_nearest_distances[i]
                    for i in range(len(k_nearest_labels))
                    if k_nearest_labels[i] == label
                ]
                avg_distances[label] = np.mean(label_distances)

            # Choose the label with the smallest average distance
            closest_label = min(avg_distances, key=avg_distances.get)
                    
            knn_out[i_test] = closest_label
        else:
            # No tie - use the most common label
            knn_out[i_test] = most_common[0][0]
#         print(knn_out)
    # Calculate accuracy
    accuracy = np.mean(knn_out == test_labels) * 100
    
    # Plot results if requested
    if plot_flag:
        plot_classification_results(knn_out, test_labels, task_num)
        
    return knn_out, accuracy
def plot_classification_results(y_pred, y_true, task_num=1):
    # Calculate accuracy
    accuracy = np.mean(y_pred == y_true) * 100
    
    # Create a figure
    plt.figure(figsize=(12, 6))
    
    # Create an array to represent if predictions were correct
    correct = y_pred == y_true
    incorrect = ~correct
    
    # Create indices for x-axis
    x = np.arange(len(y_true))
    
    # Plot correctly classified points at y=1
    plt.scatter(x[correct], np.ones_like(x[correct]), color='green', marker='o', s=80, 
                label=f'Correctly classified ({sum(correct)} samples)', alpha=0.7)
    
    # Plot incorrectly classified points at y=2
    plt.scatter(x[incorrect], np.ones_like(x[incorrect])*2, color='red', marker='x', s=80, 
                label=f'Incorrectly classified ({sum(incorrect)} samples)', alpha=0.7)
    
    # Add title and labels
    plt.title(f"Classification Results (Accuracy: {accuracy:.2f}%)")
    plt.xlabel("Test sample index")
    plt.ylabel("Classification result")
    
    # Set y-ticks to 1 and 2 with custom labels
    plt.yticks([1, 2], ['Correct', 'Incorrect'])
    
    # Set y-limits with some padding
    plt.ylim(0.5, 2.5)
    
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Task 1

In [ ]:
# Step 1: Prepare your data using your existing function
train_set, train_labels, test_set, test_labels = separate_train_test_manual(
    data=flattened_data,  
    labels=person_labels,
    sub_num_total=200,
    train_num=200,
    task_num=1  # For person identification, use 1; for expression classification, use 2
)

In [ ]:
# Step 2: Apply k-NN with different k values
k_values = [1,2,3,4,5, 6, 7, 8, 9, 10,11,12,13,14,15,16,17,18,19,20]  # Try different odd values of k
results = []

for k in k_values:
    predictions, accuracy = my_knn(
        k=k,
        train_set=train_set,
        test_set=test_set,
        train_labels=train_labels,
        test_labels=test_labels,
        task_num=1,  # Same as above
        plot_flag=(k == 5)  # Only plot for k=5 to avoid too many plots
    )
    
    results.append((k, accuracy))
    print(f"k={k}: Accuracy = {accuracy:.2f}%")
    
# Step 3: Find best k
best_k, best_accuracy = max(results, key=lambda x: x[1])
print(f"\nBest k value: {best_k} with accuracy {best_accuracy:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

# Unpack results into two separate lists
k_vals = [k for k, acc in results]
accuracies = [acc for k, acc in results]

# Plotting
plt.figure(figsize=(8, 5))
plt.plot(k_vals, accuracies, marker='o', linestyle='-', color='blue', label='k-NN Accuracy')
plt.title('k-NN Accuracy vs. k')
plt.xlabel('k (Number of Nearest Neighbors)')
plt.ylabel('Accuracy (%)')
plt.xticks(k_vals)
plt.grid(True)
plt.legend()

# Highlight best k
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best k = {best_k}')
plt.scatter(best_k, best_accuracy, color='red')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Re-split the dataset
train_set, train_labels, test_set, test_labels = separate_train_test_manual(
    data=flattened_data,
    labels=person_labels, 
    sub_num_total=200,
    train_num=200,
    task_num=1
)

# Parameters
pca_components_list = [1, 2, 3, 10, 50, 80, 100, 120, 139]
mda_components_list = [1, 2, 3, 10, 50, 80, 100, 120, 139, 150, 170, 199]
k_values = [1, 3, 5, 7, 9]

# Store results
pca_results_by_k = {k: [] for k in k_values}
mda_results_by_k = {k: [] for k in k_values}

# --- PCA ---
for num_components in pca_components_list:
    # Apply PCA
    pca_result_train, components, _, mean_vec = compute_pca(train_set, num_components=num_components)
    centered_test = test_set - mean_vec
    pca_result_test = np.dot(centered_test, components)

    # Try all k values
    for k in k_values:
        _, accuracy = my_knn(
            k=k,
            train_set=pca_result_train,
            test_set=pca_result_test,
            train_labels=train_labels,
            test_labels=test_labels,
            task_num=1,
            plot_flag=False
        )
        pca_results_by_k[k].append(accuracy)
best_pca = None
max_pca_accuracy = -1
for k in k_values:
    for i, comp in enumerate(pca_components_list):
        acc = pca_results_by_k[k][i]
        if acc > max_pca_accuracy:
            max_pca_accuracy = acc
            best_pca = (k, comp)

print(f"\n🔷 Best PCA + k-NN result: Accuracy = {max_pca_accuracy:.2f}% at k = {best_pca[0]}, components = {best_pca[1]}")
#         print(f"[PCA] Components={num_components}, k={k} → Accuracy={accuracy:.2f}%")

# --- MDA ---
for num_components in mda_components_list:
    # Apply MDA
    mda_result_train, components, _, mean_vec = compute_mda(train_set, train_labels, num_components=num_components)
    centered_test = test_set - mean_vec
    mda_result_test = np.dot(centered_test, components)

    # Try all k values
    for k in k_values:
        _, accuracy = my_knn(
            k=k,
            train_set=mda_result_train,
            test_set=mda_result_test,
            train_labels=train_labels,
            test_labels=test_labels,
            task_num=1,
            plot_flag=False
        )
        mda_results_by_k[k].append(accuracy)
#         print(f"[MDA] Components={num_components}, k={k} → Accuracy={accuracy:.2f}%")
best_mda = None
max_mda_accuracy = -1
for k in k_values:
    for i, comp in enumerate(mda_components_list):
        acc = mda_results_by_k[k][i]
        if acc > max_mda_accuracy:
            max_mda_accuracy = acc
            best_mda = (k, comp)

print(f"🔶 Best MDA + k-NN result: Accuracy = {max_mda_accuracy:.2f}% at k = {best_mda[0]}, components = {best_mda[1]}")

# --- Plotting PCA ---
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
for k in k_values:
    plt.plot(pca_components_list, pca_results_by_k[k], marker='o', label=f'k={k}')
plt.axhline(61.50, color ='black',linestyle='--', label=f'normal k-nn with K=3' )
plt.title('PCA + k-NN Accuracy')
plt.xlabel('Number of PCA Components')
plt.ylabel('Accuracy (%)')
plt.xticks(pca_components_list)
plt.grid(True)
plt.legend(title='k value')

# --- Plotting MDA ---
plt.subplot(1, 2, 2)
for k in k_values:
    plt.plot(mda_components_list, mda_results_by_k[k], marker='s', label=f'k={k}')
plt.axhline(61.50, color ='black',linestyle='--', label=f'normal k-nn with K=3' )
plt.title('MDA + k-NN Accuracy')
plt.xlabel('Number of MDA Components')
plt.ylabel('Accuracy (%)')
plt.xticks(mda_components_list)
plt.grid(True)
plt.legend(title='k value')

plt.tight_layout()
plt.show()


## Task 2

In [ ]:
# Step 1: Prepare your data using your existing function
train_set, train_labels, test_set, test_labels = separate_train_test_manual(
    data=flattened_data,  # Or use mda_result or flattened_data
    labels=person_labels,
    sub_num_total=200,
    train_num=200,
    task_num=2  # For person identification, use 1; for expression classification, use 2
)

In [ ]:
# Step 2: Apply k-NN with different k values
k_values = k_values = [1,2,3,4,5, 6, 7, 8, 9, 10,11,12,13,14,15,16,17,18,19,20]  # Try different odd values of k
results = []

for k in k_values:
    predictions, accuracy = my_knn(
        k=k,
        train_set=train_set,
        test_set=test_set,
        train_labels=train_labels,
        test_labels=test_labels,
        task_num=1,  # Same as above
        plot_flag=(k == 5)  # Only plot for k=5 to avoid too many plots
    )
    
    results.append((k, accuracy))
    print(f"k={k}: Accuracy = {accuracy:.2f}%")
    
# Step 3: Find best k
best_k, best_accuracy = max(results, key=lambda x: x[1])
print(f"\nBest k value: {best_k} with accuracy {best_accuracy:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

# Unpack results into two separate lists
k_vals = [k for k, acc in results]
accuracies = [acc for k, acc in results]

# Plotting
plt.figure(figsize=(8, 5))
plt.plot(k_vals, accuracies, marker='o', linestyle='-', color='blue', label='k-NN Accuracy')
plt.title('k-NN Accuracy vs. k')
plt.xlabel('k (Number of Nearest Neighbors)')
plt.ylabel('Accuracy (%)')
plt.xticks(k_vals)
plt.grid(True)
plt.legend()

# Highlight best k
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best k = {best_k}')
plt.scatter(best_k, best_accuracy, color='red')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Re-split the dataset
train_set, train_labels, test_set, test_labels = separate_train_test_manual(
    data=flattened_data,
    labels=person_labels, 
    sub_num_total=200,
    train_num=200,
    task_num=2
)

# Parameters
pca_components_list = [1, 2, 3, 10, 50, 80, 100, 120, 139]
mda_components_list = [1]
k_values = [1, 3, 5, 7, 13]

# Store results
pca_results_by_k = {k: [] for k in k_values}
mda_results_by_k = {k: [] for k in k_values}

# --- PCA ---
for num_components in pca_components_list:
    # Apply PCA
    pca_result_train, components, _, mean_vec = compute_pca(train_set, num_components=num_components)
    centered_test = test_set - mean_vec
    pca_result_test = np.dot(centered_test, components)

    # Try all k values
    for k in k_values:
        _, accuracy = my_knn(
            k=k,
            train_set=pca_result_train,
            test_set=pca_result_test,
            train_labels=train_labels,
            test_labels=test_labels,
            task_num=1,
            plot_flag=False
        )
        pca_results_by_k[k].append(accuracy)
best_pca = None
max_pca_accuracy = -1
for k in k_values:
    for i, comp in enumerate(pca_components_list):
        acc = pca_results_by_k[k][i]
        if acc > max_pca_accuracy:
            max_pca_accuracy = acc
            best_pca = (k, comp)

print(f"\n🔷 Best PCA + k-NN result: Accuracy = {max_pca_accuracy:.2f}% at k = {best_pca[0]}, components = {best_pca[1]}")
#         print(f"[PCA] Components={num_components}, k={k} → Accuracy={accuracy:.2f}%")

# --- MDA ---
for num_components in mda_components_list:
    # Apply MDA
    mda_result_train, components, _, mean_vec = compute_mda(train_set, train_labels, num_components=num_components)
    centered_test = test_set - mean_vec
    mda_result_test = np.dot(centered_test, components)

    # Try all k values
    for k in k_values:
        _, accuracy = my_knn(
            k=k,
            train_set=mda_result_train,
            test_set=mda_result_test,
            train_labels=train_labels,
            test_labels=test_labels,
            task_num=1,
            plot_flag=False
        )
        mda_results_by_k[k].append(accuracy)
#         print(f"[MDA] Components={num_components}, k={k} → Accuracy={accuracy:.2f}%")
best_mda = None
max_mda_accuracy = -1
for k in k_values:
    for i, comp in enumerate(mda_components_list):
        acc = mda_results_by_k[k][i]
        if acc > max_mda_accuracy:
            max_mda_accuracy = acc
            best_mda = (k, comp)

print(f"🔶 Best MDA + k-NN result: Accuracy = {max_mda_accuracy:.2f}% at k = {best_mda[0]}, components = {best_mda[1]}")

# --- Plotting PCA ---
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
for k in k_values:
    plt.plot(pca_components_list, pca_results_by_k[k], marker='o', label=f'k={k}')
plt.axhline(83.75, color ='black',linestyle='--', label=f'normal k-nn with K=9')
plt.title('PCA + k-NN Accuracy')
plt.xlabel('Number of PCA Components')
plt.ylabel('Accuracy (%)')
plt.xticks(pca_components_list)
plt.grid(True)
plt.legend(title='k value')

# --- Plotting MDA ---
plt.subplot(1, 2, 2)
for k in k_values:
    plt.plot(mda_components_list, mda_results_by_k[k], marker='s', label=f'k={k}')
plt.axhline(83.75, color ='black',linestyle='--', label=f'normal k-nn with K=9' )
plt.title('MDA + k-NN Accuracy')
plt.xlabel('Number of MDA Components')
plt.ylabel('Accuracy (%)')
plt.xticks(mda_components_list)
plt.grid(True)
plt.legend(bbox_to_anchor=(0, 0.15), loc='lower left', title='k value')

plt.tight_layout()
plt.show()